# 05-03 LangGraph 状态图编排（核心！）

**LangGraph** 是 LangChain 团队推出的 Agent 编排框架，用**有向图（State Graph）**定义 Agent 的工作流。
B站商业化 JD 明确要求 LangGraph。

**本节目标**：
- 理解 StateGraph + TypedDict 状态定义
- 掌握 Node（节点）、Edge（边）、Conditional Edge（条件边）
- 实现带循环的 Agent 工作流（生成→审核→优化→再审核）

---

In [ ]:
import os, sys
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")

try:
    from langgraph.graph import StateGraph, END
    from typing import TypedDict, Annotated
    import operator
    HAS_LG = True
    print("LangGraph 导入成功")
except ImportError:
    HAS_LG = False
    print("LangGraph 未安装: pip install langgraph")

## 1. 定义 Agent 状态

LangGraph 的核心是 **State**（状态）—— 一个 TypedDict，所有 Node 共享和修改它。

In [ ]:
from typing import TypedDict, Literal

class AdWorkflowState(TypedDict):
    """广告素材生成工作流的状态"""
    # 输入
    product_info: str                    # 产品信息
    target_audience: str                 # 目标受众
    
    # 中间状态
    draft_title: str                     # 生成的标题草稿
    draft_body: str                      # 生成的正文草稿
    review_result: str                   # 审核结果
    is_approved: bool                    # 是否通过
    iteration: int                       # 迭代次数
    
    # 输出
    final_title: str                     # 最终标题
    final_body: str                      # 最终正文

print("状态定义完成")
print("所有 Node 函数接收 state，返回需要更新的字段 dict")

## 2. 定义 Node 函数

In [ ]:
from utils.llm_client import call_llm

def generate_creative(state: AdWorkflowState) -> dict:
    """Node: 生成广告素材"""
    iteration = state.get("iteration", 0)
    feedback = state.get("review_result", "")
    
    prompt = f"""为以下产品生成B站广告素材：
产品: {state['product_info']}
目标受众: {state['target_audience']}
{f'上次审核反馈: {feedback}，请修改。' if feedback else ''}

要求：标题15字以内，正文50字以内，不含极限词。
格式：
标题：xxx
正文：xxx"""
    
    try:
        response = call_llm(prompt, max_tokens=200)
        lines = response.strip().split("\n")
        title = lines[0].replace("标题：", "").replace("标题:", "").strip()
        body = lines[-1].replace("正文：", "").replace("正文:", "").strip() if len(lines) > 1 else "精彩内容等你发现"
    except Exception:
        title = f"B站限定游戏皮肤上新{'(优化版)' if iteration > 0 else ''}"
        body = "全新皮肤登场，限时折扣体验沉浸式游戏世界"
    
    print(f"  [生成] 第{iteration+1}版: {title}")
    return {"draft_title": title, "draft_body": body, "iteration": iteration + 1}


def review_creative(state: AdWorkflowState) -> dict:
    """Node: 审核广告素材"""
    title = state["draft_title"]
    body = state["draft_body"]
    
    issues = []
    # 规则审核
    forbidden = ["最", "第一", "绝对", "100%", "万能"]
    for word in forbidden:
        if word in title or word in body:
            issues.append(f"含极限词'{word}'")
    if len(title) > 15:
        issues.append(f"标题超长({len(title)}字>15字)")
    
    is_approved = len(issues) == 0
    review = "通过" if is_approved else f"不通过: {'; '.join(issues)}"
    print(f"  [审核] {review}")
    
    if is_approved:
        return {
            "is_approved": True, "review_result": "通过",
            "final_title": title, "final_body": body
        }
    return {"is_approved": False, "review_result": "; ".join(issues)}


def route_after_review(state: AdWorkflowState) -> str:
    """条件路由: 通过→结束，不通过且<3次→重新生成，否则→结束"""
    if state.get("is_approved", False):
        return "approved"
    if state.get("iteration", 0) < 3:
        return "retry"
    return "max_retries"

print("Node 函数定义完成")

## 3. 构建状态图

In [ ]:
if HAS_LG:
    # 创建状态图
    workflow = StateGraph(AdWorkflowState)
    
    # 添加节点
    workflow.add_node("generate", generate_creative)
    workflow.add_node("review", review_creative)
    
    # 添加边
    workflow.set_entry_point("generate")       # 入口
    workflow.add_edge("generate", "review")     # 生成 → 审核
    
    # 条件边：审核后的路由
    workflow.add_conditional_edges(
        "review",
        route_after_review,
        {
            "approved":    END,          # 通过 → 结束
            "retry":       "generate",   # 不通过 → 重新生成（循环！）
            "max_retries": END,          # 超过重试次数 → 结束
        }
    )
    
    # 编译
    app = workflow.compile()
    print("状态图编译成功！")
    
    # 可视化
    try:
        print("\n状态图结构:")
        print(app.get_graph().draw_mermaid())
    except Exception:
        print("generate → review → [approved: END, retry: generate, max_retries: END]")
else:
    print("""
LangGraph 状态图构建流程:
  1. workflow = StateGraph(MyState)       # 传入状态类型
  2. workflow.add_node("name", func)      # 添加节点
  3. workflow.add_edge("a", "b")          # 添加固定边
  4. workflow.add_conditional_edges(       # 添加条件边
         "review", route_fn, {"pass": END, "fail": "generate"}
     )
  5. workflow.set_entry_point("generate")  # 设置入口
  6. app = workflow.compile()              # 编译
  7. result = app.invoke(initial_state)    # 运行
    """)

## 4. 运行工作流

In [ ]:
initial_state = {
    "product_info": "B站会员年卡，支持1080P、免广告、大会员番剧",
    "target_audience": "18-25岁动漫爱好者",
    "draft_title": "",
    "draft_body": "",
    "review_result": "",
    "is_approved": False,
    "iteration": 0,
    "final_title": "",
    "final_body": "",
}

if HAS_LG:
    print("=== 运行广告素材生成工作流 ===")
    
    # 流式执行（看到每个 node 的输出）
    for event in app.stream(initial_state):
        node_name = list(event.keys())[0]
        print(f"  [{node_name}] 完成")
    
    # 获取最终状态
    final = app.invoke(initial_state)
    print(f"\n=== 最终结果 ===")
    print(f"迭代次数: {final.get('iteration', 0)}")
    print(f"通过审核: {final.get('is_approved', False)}")
    print(f"最终标题: {final.get('final_title', final.get('draft_title', 'N/A'))}")
    print(f"最终正文: {final.get('final_body', final.get('draft_body', 'N/A'))}")
else:
    # 手动模拟工作流
    print("=== 模拟运行工作流 ===")
    state = dict(initial_state)
    for i in range(3):
        state.update(generate_creative(state))
        state.update(review_creative(state))
        if state["is_approved"]:
            break
    
    print(f"\n最终: 标题='{state.get('final_title', state['draft_title'])}', "
          f"通过={state['is_approved']}, 迭代={state['iteration']}")

## LangGraph 核心概念速查

| 概念 | 说明 |
|------|------|
| `StateGraph` | 有状态的有向图，所有 node 共享一个 state |
| `Node` | 节点 = 一个 Python 函数，接收 state 返回 partial state |
| `Edge` | 固定边 = 无条件从 A 到 B |
| `Conditional Edge` | 条件边 = 根据 state 动态选择下一个节点 |
| `END` | 特殊节点，表示工作流结束 |
| `Reducer` | 当多个 node 更新同一字段时的合并策略(如 operator.add) |
| `Checkpointer` | 状态持久化，支持暂停/恢复/回放 |

## 面试速记

| 问题 | 要点 |
|------|------|
| LangGraph vs AgentExecutor | LangGraph 用图定义控制流，更灵活（支持循环、分支、并行）；AgentExecutor 是简单的 while 循环 |
| 为什么需要状态图 | 复杂 Agent 需要条件分支、循环重试、人工审核等，简单 while 循环无法表达 |
| Cycle vs DAG | LangGraph 支持 Cycle（循环），DAG 不支持；循环 = 重试/迭代优化 |

**下一节**: `04_langgraph_multi_agent.ipynb`